[ PYTORCH MODEL ]

- 주제 : 펭귄 품종 분류 모델
- 데이터 :  penguins.csv
- 구성 : 품종 + 피쳐

In [126]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import *

In [127]:
data_file = '../_data/penguins.csv'

In [128]:
# [2] 데이터로딩및 확인

In [129]:
dataDF = pd.read_csv(data_file)

In [130]:
dataDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [131]:
## 결측치 컬럼이 7개중 5개 존재
dataDF.dropna(inplace=True)

In [132]:
dataDF.isna().sum()

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

[3-2] 전처리 

In [133]:
dataDF['sex'] = dataDF['sex'].replace({'MALE':0, 'FEMALE':1})
dataDF['species'] = dataDF['species'].replace({x:y for x, y in zip(dataDF['species'].unique(), range(len(dataDF['species'].unique())))})
dataDF['island'] = dataDF['island'].replace({x:y for x, y in zip(dataDF['island'].unique(), range(len(dataDF['island'].unique())))})


C:\Users\kdt\AppData\Local\Temp\ipykernel_27116\3629993404.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataDF['sex'] = dataDF['sex'].replace({'MALE':0, 'FEMALE':1})
C:\Users\kdt\AppData\Local\Temp\ipykernel_27116\3629993404.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataDF['species'] = dataDF['species'].replace({x:y for x, y in zip(dataDF['species'].unique(), range(len(dataDF['species'].unique())))})
C:\Users\kdt\AppData\Local\Temp\ipykernel_27116\3629993404.py:3: FutureWarning: Downcasting behavior in `replace` i

In [134]:
dataDF['species']

0      0
1      0
2      0
4      0
5      0
      ..
338    2
340    2
341    2
342    2
343    2
Name: species, Length: 333, dtype: int64

In [135]:
dataDF = dataDF.reset_index(drop=True)

In [136]:
dataDF.dtypes

species                int64
island                 int64
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                    int64
dtype: object

In [137]:
# from sklearn.preprocessing import OneHotEncoder

In [138]:
# ohe = OneHotEncoder()
# encoded_payment = ohe.fit_transform(dataDF[['species']]).toarray()
# dataDF= pd.concat((dataDF, pd.DataFrame(encoded_payment, columns=ohe.get_feature_names_out(['species']))), axis=1)

In [139]:
# ohe = OneHotEncoder()
# encoded_payment = ohe.fit_transform(dataDF[['island']]).toarray()
# dataDF= pd.concat((dataDF, pd.DataFrame(encoded_payment, columns=ohe.get_feature_names_out(['island']))), axis=1)

In [140]:
# 

In [141]:
# dataDF

In [142]:
# 3-3 타겟 기준 상관관계
# sns.PairGrid(dataDF)
# bill_length_mm bill_depth_mm flipper_length_mm body_mass_g 
dataDF.corr(numeric_only=True)['species']

species              1.000000
island              -0.009176
bill_length_mm       0.730548
bill_depth_mm       -0.740346
flipper_length_mm    0.850737
body_mass_g          0.750434
sex                 -0.010964
Name: species, dtype: float64

In [143]:
# sns.pairplot(dataDF.loc[:,:])

In [144]:
# 3-4 피쳐간 관계성
## 수치확인
featureDF = dataDF.loc[:, [ 'bill_length_mm', 'bill_depth_mm','flipper_length_mm', 'body_mass_g',]]
featureDF.corr()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.000000,-0.228626,0.653096,0.589451
bill_depth_mm,-0.228626,1.000000,-0.577792,-0.472016
flipper_length_mm,0.653096,-0.577792,1.000000,0.872979
body_mass_g,0.589451,-0.472016,0.872979,1.000000


In [145]:
## 수치형 피쳐 스케일링
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
mmscale = MinMaxScaler()
scaleNP = mmscale.fit_transform(featureDF)
scaleDF = pd.DataFrame(scaleNP)
scaleDF.head()

,0,1,2,3
0,0.254545,0.666667,0.152542,0.291667
1,0.269091,0.511905,0.237288,0.305556
2,0.298182,0.583333,0.389831,0.152778
3,0.167273,0.738095,0.355932,0.208333
4,0.261818,0.892857,0.305085,0.263889


### 신경망모델 구현 <hr>
- 피쳐와 타겟 분리
- 피쳐와 타겟 가공
- 학습용 & 테스트
- 학습용 기준으로 표준화 & 정규화 진행
- 데이터셋 클래스 설계 및 구현
- 모델 클래스 설계 및 구현
- 학습 진행
- 학습 결과 분석
- 튜닝 여부 결정
- 모델 저장 & 서비스 연동

In [146]:
import torch         
# tensor 및 기본 함수 모듈   
import torch.nn as nn
# 인공신경망 관련 모듈
import torch.nn.functional as F
# 인공신경망 관련 함수
import torch.optim as optim
# 최적화 모듈
from torchmetrics.classification import *           # 데이터셋 관련
from torch.utils.data import Dataset, DataLoader    # 모델 성능지표 관련
from torchinfo import summary                       # 모델 구조 및 정보 관련

import sys

[2] 데이터 피쳐와 타겟 분리 <hr>

In [147]:
sys.path.append('../_utils/')

In [148]:
dataDF = pd.read_csv(data_file)
dataDF.dropna(inplace=True)

In [149]:
# 피처와 타겟 분리
featureDF = dataDF.loc[:, [ 'bill_length_mm', 'bill_depth_mm','flipper_length_mm', 'body_mass_g',]]
targetSR = dataDF[dataDF.columns[0]]

In [150]:
print(f"featureDF : {featureDF.shape} targetSR: {targetSR.shape}")

featureDF : (333, 4) targetSR: (333,)


In [151]:
import tools as ts
X_train, X_test, y_train, y_test = ts.train_test_cut(featureDF, targetSR, stratify=targetSR, RandomState=42)

X_train => 2D (249, 4) / X_test => 2D, (84, 4)
y_train => 1D (249,), / y_test => 1D, (84,)


[4] 학습용 기준으로 정규화 진행 <hr>

In [152]:
mmScaler = MinMaxScaler()
mmScaler.fit(X_train)

scale_X_train = mmScaler.transform(X_train)
scale_X_test = mmScaler.transform(X_test)

In [154]:
lbEncoder = LabelEncoder()
lbEncoder.fit(y_train)
y_train = lbEncoder.transform(y_train)
y_test = lbEncoder.transform(y_test)

In [ ]:
## 펭귄데이터에 대한 전용 Dataset 클래스 정의
## 클래스이름: PenguineDS
## 데이터 구성: 피쳐3 DF + 타겟 1 DF
## 필수메서드 : __init__(self, 피쳐DF, 타겟DF)
##              __len__(self) => return 샘플 수
##              __getitem__(self, index) => return index에 해당하는 feature, target => tenosr값.

class PenguineDS(nn.Module):
    
    def __init__(self, featureDF, targetDF):
        super().__init__()
        self.feature = featureDF
        self.target = targetDF
        self.n_features = featureDF.shape[1]
        self.n_samples = featureDF.shape[0]
        
    def __len__(self):
        return self.n_samples        
        
    def __getitem__(self, index):
        farr = self.feature.iloc[index].values
        tarr = self.target.iloc[index].values
        
        return torch.FloatTensor(farr), torch.LongTensor(tarr)
        
        

In [174]:
fDF = pd.DataFrame(scale_X_train)
tDF = pd.Series(y_train).to_frame()

testDS = PenguineDS(fDF, tDF)

In [179]:
for feature, target  in testDS:
    print(feature, target)
    break

tensor([0.6705, 0.7037, 0.5254, 0.3529]) tensor([1.])


In [177]:
## 펭귄데이터에 대한 전용 Dataset 클래스 정의
## 클래스이름: PenguineNPDS
## 데이터 구성: 피쳐3 NP + 타겟 1 NP
## 필수메서드 : __init__(self, 피쳐DF, 타겟DF)
##              __len__(self) => return 샘플 수
##              __getitem__(self, index) => return index에 해당하는 feature, target => tenosr값.

class PenguineDS(nn.Module):
    
    def __init__(self, featureNP, targetNP):
        super().__init__()
        self.feature = featureNP
        self.target = targetNP
        self.n_features = featureNP.shape[1]
        self.n_samples = featureNP.shape[0]
        
    def __len__(self):
        return self.n_samples        
        
    def __getitem__(self, index):
        return torch.FloatTensor(self.feature[index]), torch.LongTensor(self.target[index])
        
        